# Custom sklearn transformer

Transformer to obiekt z dwoma metodami:
- `fit(X)`      : naucz się parametrów z danych treningowych, zwróć `self`
- `transform(X)`: zastosuj nauczonych parametrów do dowolnych danych

Implementując ten kontrakt możemy wrzucić własny krok do `Pipeline`.

In [ ]:
import numpy as np
import pandas as pd

# Dane treningowe: normalne wartości, bez outlierów
df_train = pd.DataFrame({
    "price":     [20.0, 35.0, 15.0, 28.0, 10.0, 22.0],
    "days_late": [ 0.0,  2.0,  0.0,  1.0,  0.0,  3.0],
    "is_bad":    [    0,    0,    0,    0,    0,    0],
})

# Dane testowe: zawierają outliery których model nie widział w treningu
df_test = pd.DataFrame({
    "price":     [18.0, 400.0],   # 400 to outlier spoza rozkładu treningowego
    "days_late": [ 1.0,  10.0],   # 10 też
    "is_bad":    [    0,     1],
})

print("TRAIN:")
print(df_train)
print()
print("TEST (zawiera outliery):")
print(df_test)

## Implementacja: OutlierClipper

`fit()` zapamiętuje percentyle z danych treningowych.
`transform()` przycina wartości do nauczonych granic.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class OutlierClipper(BaseEstimator, TransformerMixin):

    def __init__(self, lower_q=0.05, upper_q=0.95):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.lower_ = np.percentile(X, self.lower_q * 100, axis=0)
        self.upper_ = np.percentile(X, self.upper_q * 100, axis=0)
        return self  # zawsze self

    def transform(self, X):
        return np.clip(np.asarray(X), self.lower_, self.upper_)

In [ ]:
X_train = df_train[["price", "days_late"]].values
y_train = df_train["is_bad"].values
X_test  = df_test[["price", "days_late"]].values
y_test  = df_test["is_bad"].values

clipper = OutlierClipper()
clipper.fit(X_train)   # uczy się tylko na train: outlier z testu nie wpływa na granice

print("Nauczone granice (z samego train):")
print(f"  price:     [{clipper.lower_[0]:.1f}, {clipper.upper_[0]:.1f}]")
print(f"  days_late: [{clipper.lower_[1]:.1f}, {clipper.upper_[1]:.1f}]")
print()

X_test_c = clipper.transform(X_test)

print("X_test przed clippingiem:", X_test[:, 0], X_test[:, 1])
print("X_test po clippingu:     ", X_test_c[:, 0], X_test_c[:, 1])
print()
print("→ 400 i 10 zostały przycięte do granic nauczonych z train")

## Pipeline: OutlierClipper → StandardScaler → model

`fit()` na Pipeline: każdy krok dostaje dane już przetworzone przez poprzedni.
`predict()` na Pipeline: te same transformacje, te same parametry z fit.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("clipper", OutlierClipper(lower_q=0.05, upper_q=0.95)),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression()),
])

pipe.fit(X_train, y_train)
print("Predykcje:", pipe.predict(X_test))
print("Prawdziwe:", y_test)
print()
# Parametry każdego kroku dostępne przez nazwę
print("Granice z Pipeline:", pipe["clipper"].lower_, pipe["clipper"].upper_)